<a href="https://colab.research.google.com/github/SAINIDHI2005/IDS_GNN_Repo/blob/main/graphSAGE/FlowFlow_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-geometric -q

In [2]:
import os

import pandas as pd
import numpy as np

import torch

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from torch_geometric.data import Data

In [3]:
DATASET_DIR = "/Users/balji/OneDrive/GNN_IDS/Balanced_6Class_400K"

files = {
    "BENIGN.csv":0,
    "DDoS.csv":1,
    "DoS.csv":1,
    "Mirai.csv":1,
    "Recon.csv":1,
    "Spoofing.csv":1
}

dfs = []

for file,label in files.items():

    path = os.path.join(DATASET_DIR,file)

    df = pd.read_csv(path)

    attack_name = file.replace(".csv","")

    df["attack_type"] = attack_name

    df["label"] = label

    dfs.append(df)

data = pd.concat(
    dfs,
    ignore_index=True
)

print(data.shape)

print(data["attack_type"].value_counts())

(400000, 88)
attack_type
BENIGN      200000
DDoS         40000
DoS          40000
Mirai        40000
Recon        40000
Spoofing     40000
Name: count, dtype: int64


In [4]:
data = data.sort_values(
    "bidirectional_first_seen_ms"
).reset_index(drop=True)

data["flow_id"] = np.arange(len(data))

print(data.shape)

(400000, 89)


In [5]:
attack_type_backup = data["attack_type"].copy()

In [6]:
DROP_COLS = [

    "id",
    "expiration_id",

    "src_ip",
    "dst_ip",

    "src_mac",
    "dst_mac",

    "src_oui",
    "dst_oui",

    "requested_server_name",

    "client_fingerprint",
    "server_fingerprint",

    "user_agent",
    "content_type",

    "application_name",
    "application_category_name",

    "flow_id",
    "label",
    "attack_type",

    "bidirectional_first_seen_ms",
    "bidirectional_last_seen_ms",

    "src2dst_first_seen_ms",
    "src2dst_last_seen_ms",

    "dst2src_first_seen_ms",
    "dst2src_last_seen_ms",

    "application_confidence",
    "application_is_guessed",

    "vlan_id",
    "tunnel_id",

    "bidirectional_urg_packets",
    "src2dst_urg_packets",
    "dst2src_urg_packets",

    "dst2src_cwr_packets",
    "dst2src_ece_packets",

    "src_port",

    "bidirectional_duration_ms",
    "bidirectional_packets",

    "src2dst_packets",
    "dst2src_packets",

    "dst2src_min_ps",
    "dst2src_mean_ps",

    "bidirectional_min_piat_ms",
    "bidirectional_stddev_piat_ms",

    "dst2src_min_piat_ms",
    "dst2src_mean_piat_ms",
    "dst2src_max_piat_ms",

    "bidirectional_ece_packets",
    "bidirectional_psh_packets",
    "bidirectional_fin_packets",

    "src2dst_syn_packets",
    "src2dst_cwr_packets",
    "src2dst_ack_packets",
    "src2dst_rst_packets",
    "src2dst_fin_packets",

    "src2dst_mean_piat_ms",
    "src2dst_stddev_ps",

    "dst2src_ack_packets",
    "dst2src_psh_packets",
    "dst2src_rst_packets",
    "dst2src_fin_packets",

]

feature_cols = [

    c
    for c in data.columns
    if c not in DROP_COLS
]

print("Features:",len(feature_cols))

Features: 30


In [7]:
feature_cols = [
    c
    for c in data.columns
    if c not in DROP_COLS
]

print("Features:", len(feature_cols))

print("\nRetained Features:\n")
for i, f in enumerate(feature_cols, start=1):
    print(f"{i:2d}. {f}")

Features: 30

Retained Features:

 1. dst_port
 2. protocol
 3. ip_version
 4. bidirectional_bytes
 5. src2dst_duration_ms
 6. src2dst_bytes
 7. dst2src_duration_ms
 8. dst2src_bytes
 9. bidirectional_min_ps
10. bidirectional_mean_ps
11. bidirectional_stddev_ps
12. bidirectional_max_ps
13. src2dst_min_ps
14. src2dst_mean_ps
15. src2dst_max_ps
16. dst2src_stddev_ps
17. dst2src_max_ps
18. bidirectional_mean_piat_ms
19. bidirectional_max_piat_ms
20. src2dst_min_piat_ms
21. src2dst_stddev_piat_ms
22. src2dst_max_piat_ms
23. dst2src_stddev_piat_ms
24. bidirectional_syn_packets
25. bidirectional_cwr_packets
26. bidirectional_ack_packets
27. bidirectional_rst_packets
28. src2dst_ece_packets
29. src2dst_psh_packets
30. dst2src_syn_packets


In [8]:
top30 = {
    "bidirectional_mean_ps",
    "dst2src_syn_packets",
    "src2dst_min_ps",
    "ip_version",
    "src2dst_max_piat_ms",
    "dst2src_stddev_piat_ms",
    "bidirectional_max_piat_ms",
    "bidirectional_rst_packets",
    "src2dst_max_ps",
    "src2dst_duration_ms",
    "src2dst_bytes",
    "src2dst_ece_packets",
    "bidirectional_max_ps",
    "dst2src_bytes",
    "bidirectional_bytes",
    "dst2src_max_ps",
    "bidirectional_min_ps",
    "bidirectional_syn_packets",
    "dst2src_stddev_ps",
    "bidirectional_ack_packets",
    "protocol",
    "dst_port",
    "bidirectional_mean_piat_ms",
    "dst2src_duration_ms",
    "src2dst_mean_ps",
    "src2dst_min_piat_ms",
    "bidirectional_stddev_ps",
    "src2dst_stddev_piat_ms",
    "bidirectional_cwr_packets",
    "src2dst_psh_packets"
}

retained = set(feature_cols)

print("Missing from retained:")
print(sorted(top30 - retained))

print("\nExtra retained features:")
print(sorted(retained - top30))

Missing from retained:
[]

Extra retained features:
[]


In [9]:
for col in feature_cols:

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data[feature_cols] = (
    data[feature_cols]
    .fillna(0)
)

In [10]:
scaler = StandardScaler()

X = scaler.fit_transform(
    data[feature_cols]
)

print(X.shape)

(400000, 30)


In [11]:
num_flow_nodes = len(data)

print("Flow Nodes:", num_flow_nodes)

Flow Nodes: 400000


In [12]:
from tqdm import tqdm

print("Building Flow-Flow Graph with Temporal Constraint...")

TIME_WINDOW_MS = 100      # 2-second temporal window

source_nodes = []
target_nodes = []

# Group by communication profile
grouped = data.groupby(
    ["protocol", "dst_ip", "dst_port"]
)

for _, group in tqdm(grouped, desc="Creating Temporal Edges"):

    # Sort flows chronologically
    group = group.sort_values("bidirectional_first_seen_ms")

    # FIX: Use 'flow_id' instead of 'id' to ensure 0-indexed, sequential node IDs
    node_ids = group["flow_id"].values
    timestamps = group["bidirectional_first_seen_ms"].values

    n = len(node_ids)

    # Sliding Window
    left = 0

    for right in range(n):

        while timestamps[right] - timestamps[left] > TIME_WINDOW_MS:
            left += 1

        # Connect current flow with previous flows inside window
        for i in range(left, right):

            source_nodes.append(node_ids[right])
            target_nodes.append(node_ids[i])

            source_nodes.append(node_ids[i])
            target_nodes.append(node_ids[right])

edge_index = torch.tensor(
    [source_nodes, target_nodes],
    dtype=torch.long
)

print("\nEdge construction complete.")
print(edge_index.shape)

Building Flow-Flow Graph with Temporal Constraint...


Creating Temporal Edges: 100%|██████████| 57299/57299 [00:19<00:00, 2905.95it/s]



Edge construction complete.
torch.Size([2, 4519560])


In [13]:
x = torch.tensor(
    X,
    dtype=torch.float
)

print(x.shape)

torch.Size([400000, 30])


In [14]:
y = torch.tensor(
    data["label"].values,
    dtype=torch.long
)

print(y.shape)

torch.Size([400000])


In [15]:
train_idx, val_idx = train_test_split(
    np.arange(num_flow_nodes),
    test_size=0.20,
    stratify=data["label"],
    random_state=42
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))

Train: 320000
Validation: 80000


In [16]:
total_nodes = num_flow_nodes

In [17]:
train_mask = torch.zeros(total_nodes,dtype=torch.bool)

val_mask = torch.zeros(total_nodes,dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True

print("Train mask:", train_mask.sum().item())
print("Validation mask :",val_mask.sum().item())

Train mask: 320000
Validation mask : 80000


In [18]:
graph = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask
)
print(graph)

print(
    "Nodes:",
    graph.num_nodes
)

print(
    "Edges:",
    graph.num_edges
)

Data(x=[400000, 30], edge_index=[2, 4519560], y=[400000], train_mask=[400000], val_mask=[400000])
Nodes: 400000
Edges: 4519560


In [19]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_classes
    ):

        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            hidden_channels
        )

        self.conv3 = SAGEConv(
            hidden_channels,
            hidden_channels // 2
        )

        self.bn1 = torch.nn.BatchNorm1d(
            hidden_channels
        )

        self.bn2 = torch.nn.BatchNorm1d(
            hidden_channels
        )

        self.bn3 = torch.nn.BatchNorm1d(
            hidden_channels // 2
        )

        self.classifier = torch.nn.Linear(
            hidden_channels // 2,
            num_classes
        )

    def forward(
        self,
        x,
        edge_index
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = self.bn1(x)

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.4,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index
        )

        x = self.bn2(x)

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.4,
            training=self.training
        )

        x = self.conv3(
            x,
            edge_index
        )

        x = self.bn3(x)

        x = F.relu(x)

        x = self.classifier(x)

        return x


In [20]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [21]:
model = GraphSAGE(
    in_channels=graph.num_node_features,
    hidden_channels=256,
    num_classes=2
)

model = model.to(device)

graph = graph.to(device)

print(model)

GraphSAGE(
  (conv1): SAGEConv(30, 256, aggr=mean)
  (conv2): SAGEConv(256, 256, aggr=mean)
  (conv3): SAGEConv(256, 128, aggr=mean)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [22]:
from torch_geometric.loader import NeighborLoader

train_loader = NeighborLoader(
    graph,
    input_nodes=graph.train_mask,
    num_neighbors=[15, 10],
    batch_size=2048,
    shuffle=True
)

val_loader = NeighborLoader(
    graph,
    input_nodes=graph.val_mask,
    num_neighbors=[15, 10],
    batch_size=2048,
    shuffle=False
)

print("Train batches :", len(train_loader))
print("Validation batches :", len(val_loader))

Train batches : 157
Validation batches : 40


In [23]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.002,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=20,
    gamma=0.5
)

In [24]:
weights = torch.tensor(
    [1.0, 1.5],
    dtype=torch.float
).to(device)

criterion = torch.nn.CrossEntropyLoss(
    weight=weights
)

In [25]:
def train():

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(
            batch.x,
            batch.edge_index
        )

        loss = criterion(
            out[:batch.batch_size],
            batch.y[:batch.batch_size]
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [26]:
@torch.no_grad()

def evaluate(loader):

    model.eval()

    correct = 0
    total = 0

    for batch in loader:

        batch = batch.to(device)

        out = model(
            batch.x,
            batch.edge_index
        )

        pred = out[:batch.batch_size].argmax(dim=1)

        correct += (
            pred ==
            batch.y[:batch.batch_size]
        ).sum().item()

        total += batch.batch_size

    return correct / total

In [27]:
import os
import time
import joblib
import torch

best_val_acc = 0.0
best_epoch = 0

SAVE_DIR = r"/Users/balji/OneDrive/GNN_IDS/Saved_Model"
os.makedirs(SAVE_DIR, exist_ok=True)

start_time = time.time()

for epoch in range(1, 151):

   loss = train()

   train_acc = evaluate(train_loader)

   val_acc = evaluate(val_loader)

   # Save best model based on validation accuracy
   if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            os.path.join(
                SAVE_DIR,
                "FlowFlow_GraphSAGE.pth"
            )
        )

        joblib.dump(
            scaler,
            os.path.join(
                SAVE_DIR,
                "FlowFlow_GraphSAGE_scaler.pkl"
            )
        )
   scheduler.step()

   print(
        f"Epoch {epoch:03d}"
        f" | Loss {loss:.4f}"
        f" | Train {train_acc:.4f}"
        f" | Validation {val_acc:.4f}"
   )

end_time = time.time()

training_time = end_time - start_time

print("\n" + "=" * 60)
print(f"Training Time          : {training_time:.2f} seconds")
print(f"Best Epoch             : {best_epoch}")
print(f"Best Validation Acc.   : {best_val_acc:.4f}")
print(f"Model Saved At         : {os.path.join(SAVE_DIR,'FlowFlow_GraphSAGE.pth')}")
print(f"Scaler Saved At        : {os.path.join(SAVE_DIR,'FlowFlow_GraphSAGE_scaler.pkl')}")
print("=" * 60)
print("Training Complete.")

Epoch 001 | Loss 0.4313 | Train 0.8310 | Validation 0.8256
Epoch 002 | Loss 0.3658 | Train 0.8526 | Validation 0.8390
Epoch 003 | Loss 0.3371 | Train 0.8579 | Validation 0.8477
Epoch 004 | Loss 0.3224 | Train 0.8611 | Validation 0.8487
Epoch 005 | Loss 0.3101 | Train 0.8688 | Validation 0.8569
Epoch 006 | Loss 0.3041 | Train 0.8788 | Validation 0.8665
Epoch 007 | Loss 0.2990 | Train 0.8742 | Validation 0.8638
Epoch 008 | Loss 0.2931 | Train 0.8730 | Validation 0.8743
Epoch 009 | Loss 0.2905 | Train 0.8809 | Validation 0.8701
Epoch 010 | Loss 0.2883 | Train 0.8848 | Validation 0.8763
Epoch 011 | Loss 0.2834 | Train 0.8864 | Validation 0.8867
Epoch 012 | Loss 0.2831 | Train 0.8872 | Validation 0.8766
Epoch 013 | Loss 0.2796 | Train 0.8895 | Validation 0.8893
Epoch 014 | Loss 0.2791 | Train 0.8884 | Validation 0.8891
Epoch 015 | Loss 0.2777 | Train 0.8894 | Validation 0.8783
Epoch 016 | Loss 0.2768 | Train 0.8900 | Validation 0.8772
Epoch 017 | Loss 0.2736 | Train 0.8885 | Validation 0.87

In [28]:
import time
import torch

print("=" * 60)

# Training Time
print(f"Training Time: {training_time:.2f} seconds ({150} epochs)\n")

# GPU Memory
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved = torch.cuda.memory_reserved() / (1024 ** 2)
    max_reserved = torch.cuda.max_memory_reserved() / (1024 ** 2)

    print(f"Allocated GPU Memory: {allocated:.2f} MB")
    print(f"Reserved GPU Memory: {reserved:.2f} MB")
    print(f"Max GPU Memory: {max_reserved:.2f} MB\n")

# Model Parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters: {total_params}")
print(f"Trainable Parameters: {trainable_params}\n")

# Graph Memory
node_memory = (
    graph.x.element_size()
    * graph.x.nelement()
) / (1024 ** 2)

edge_memory = (
    graph.edge_index.element_size()
    * graph.edge_index.nelement()
) / (1024 ** 2)

graph_memory = node_memory + edge_memory

print(f"Node Feature Memory: {node_memory:.2f} MB")
print(f"Edge Memory: {edge_memory:.2f} MB")
print(f"Total Graph Memory: {graph_memory:.2f} MB")

print("=" * 60)

Training Time: 2224.45 seconds (150 epochs)

Allocated GPU Memory: 140.17 MB
Reserved GPU Memory: 596.00 MB
Max GPU Memory: 596.00 MB

Total Parameters: 214146
Trainable Parameters: 214146

Node Feature Memory: 45.78 MB
Edge Memory: 68.96 MB
Total Graph Memory: 114.74 MB
